# Lab 9: Clinical NLP - Classifying Medical Specialties from Transcriptions

**BINF 5507 - Applications III: Clinical Natural Language Processing**

In earlier labs we worked with tidy, tabular data. Real clinical data is often the opposite: roughly 80% of what lives in an electronic health record (EHR) is **unstructured free text** - progress notes, discharge summaries, radiology and pathology reports.

In this lab, we will build a pipeline that turns messy clinical transcriptions into features a machine-learning model can use, and then predict the **medical specialty** a note came from. This is the same conceptual pipeline from the lecture:

> **raw text -> preprocess -> represent (features) -> classify -> evaluate**

**Learning objectives**
1. Explore and clean unstructured clinical text (callback to Lab 2).
2. Represent text numerically with **TF-IDF**.
3. Train and compare text classifiers (callback to Lab 4).
4. Evaluate honestly under **class imbalance** (accuracy vs. macro-F1).
5. See where classical NLP ends and **embeddings / LLMs** begin.

**Dataset:** `mtsamples.csv` - 4,999 transcribed medical reports across 40 specialties, scraped from mtsamples.com (CC0 public domain).

## 0. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 200)

# The CSV lives in the same folder as this notebook
df = pd.read_csv("mtsamples.csv", index_col=0)
print("Raw shape:", df.shape)
df.head(3)

In [ ]:
print("Columns:", list(df.columns))

# Look at ONE full transcription so you can see what clinical text really looks like
print(df["transcription"].iloc[0])

**Column guide**
- `description` - a one-line summary of the report
- `medical_specialty` - the label we will try to predict
- `sample_name` - short title of the report
- `transcription` - the full note text (our main input)
- `keywords` - manually assigned keywords (a possible source of leakage - see Questions)

#### Questions for Exploration
1. Skim the transcription printed above. What structure do you notice (section headers, punctuation, abbreviations)? Why might this be harder to model than the tabular data from Lab 4?
2. The `keywords` column often literally contains the specialty name. If we trained on `keywords`, why would our accuracy be misleadingly high? (This is a classic **data leakage** trap.)

## 1. Exploring the Text (EDA)

In [ ]:
# Specialty labels have leading/trailing whitespace - strip it first
df["medical_specialty"] = df["medical_specialty"].str.strip()

print("Number of specialties:", df["medical_specialty"].nunique())
counts = df["medical_specialty"].value_counts()
print(counts.head(15))

In [ ]:
# Visualise the class distribution
plt.figure(figsize=(9, 6))
counts.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 specialties by number of reports")
plt.xlabel("Number of reports")
plt.tight_layout()
plt.show()

In [ ]:
# How much text is in each note? (word count distribution)
df["n_words"] = df["transcription"].fillna("").str.split().apply(len)
df["n_words"].describe()

In [ ]:
# Some transcriptions are missing - we cannot model an empty note
print("Missing transcriptions:", df["transcription"].isna().sum())
df = df.dropna(subset=["transcription"]).reset_index(drop=True)
print("Rows after dropping missing:", len(df))

#### Questions for Exploration
1. The class distribution is heavily **imbalanced** (`Surgery` has ~1,100 reports; several specialties have fewer than 20). If a model just always guessed "Surgery", roughly what accuracy would it get? Why does that make plain accuracy a poor metric here?
2. Look at the top categories. Which of these are true clinical *specialties* and which are really *document types* (e.g., `Consult - History and Phy.`, `SOAP / Chart / Progress Notes`)? We will deal with this in Section 3.

## 2. Text Preprocessing (callback to Lab 2)

Clinical transcriptions in this dataset use commas as inline section delimiters (e.g. `,MEDICATIONS:,`) and are full of punctuation and numbers. Before we can build features we normalise the text: lowercase it, remove the comma-delimiters and non-letter characters, and collapse extra whitespace.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r",", " ", text)          # inline comma delimiters -> spaces
    text = re.sub(r"[^a-z\s]", " ", text)   # drop digits and punctuation
    text = re.sub(r"\s+", " ", text).strip() # collapse whitespace
    return text

# Compare a raw vs. cleaned snippet
raw = df["transcription"].iloc[0]
print("RAW:\n", raw[:300])
print("\nCLEANED:\n", clean_text(raw)[:300])

In [ ]:
# Apply cleaning to every note (creates a new 'clean' column)
df["clean"] = df["transcription"].apply(clean_text)
df[["medical_specialty", "clean"]].head(3)

#### Questions for Exploration
1. Our cleaner deletes **all digits**. For predicting specialty that is probably fine - but name a clinical NLP task where throwing away numbers would be a serious mistake.
2. Generic English stop-word lists remove words like "no" and "not". In clinical text, why is deleting negation words potentially dangerous? (Think: "no evidence of malignancy".)

## 3. Scoping the Problem

Trying to classify all 40 categories at once is hard and noisy: several categories are document *types* rather than specialties, and the smallest classes have too few examples to learn from. We will focus on **10 genuinely distinct clinical specialties**. This keeps the problem tractable and the results interpretable - a deliberate modelling choice, not cheating.

In [ ]:
keep = [
    "Cardiovascular / Pulmonary", "Orthopedic", "Radiology",
    "Gastroenterology", "Neurology", "Obstetrics / Gynecology",
    "Urology", "Nephrology", "ENT - Otolaryngology", "Ophthalmology",
]
data = df[df["medical_specialty"].isin(keep)].copy().reset_index(drop=True)
print("Scoped rows:", len(data), "| classes:", data["medical_specialty"].nunique())
data["medical_specialty"].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split so every specialty appears in both train and test in the same proportion
X_train, X_test, y_train, y_test = train_test_split(
    data["clean"], data["medical_specialty"],
    test_size=0.2, stratify=data["medical_specialty"], random_state=42,
)
print("Train:", len(X_train), "| Test:", len(X_test))

#### Questions for Exploration
1. We passed `stratify=data["medical_specialty"]`. What could go wrong on this imbalanced dataset if we did a plain random split instead?
2. We reduced 40 classes to 10. What kind of note would this model have *no chance* of classifying correctly now, and is that an acceptable trade-off for a teaching example?

## 4. Feature Representation: TF-IDF

A model cannot read text - we must turn each note into a vector of numbers. **TF-IDF** (Term Frequency-Inverse Document Frequency) weights each word by how often it appears in a note, down-weighted by how common it is across *all* notes. Distinctive clinical terms (e.g. "coronary", "fracture") get high weights; ubiquitous words get low weights.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# `stop_words``="english" removes common English words
# `max_features`` caps the vocabulary size (try 5000);
# `ngram_range``=(1, 2) captures single words AND two-word phrases.
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
)

# Fit only on training data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("Feature matrix shape (train):", X_train_tfidf.shape)


In [ ]:
# Peek at the highest-weighted terms overall (sanity check)
import numpy as np
feature_names = np.array(vectorizer.get_feature_names_out())
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top = feature_names[mean_tfidf.argsort()[::-1][:20]]
print("Top 20 terms by mean TF-IDF:\n", list(top))

#### Questions for Exploration
1. What is the shape of your feature matrix (rows x columns)? Compare the number of features to the number of training notes - this **high-dimensional, few-sample** situation is exactly the "curse of dimensionality" from the imaging lecture.
2. Re-run with `ngram_range=(1, 1)` (single words only) vs. `(1, 2)`. How does the vocabulary size change, and why might bigrams like "chest pain" help?

## 5. Classification (callback to Lab 4)

Now we train classifiers on the TF-IDF features. As in Lab 4 we compare several models. Because the classes are imbalanced, we pass `class_weight="balanced"` where the model supports it (Naive Bayes does not).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced"),
}

results = {}
for name, model in models.items():
    # fit the model on X_train_tfidf / y_train
    model.fit(X_train_tfidf, y_train)
    # predict on X_test_tfidf
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average="macro")
    results[name] = {"model": model, "preds": preds, "acc": acc, "macro_f1": macro_f1}
    print(f"{name:26s}  accuracy={acc:.3f}  macro-F1={macro_f1:.3f}")

#### Questions for Exploration
1. Which model performs best on **macro-F1**? Is it the same model that wins on accuracy? Why can these disagree?
2. Naive Bayes and linear models are strong baselines for text. Why do simple linear models tend to work well on high-dimensional, sparse TF-IDF features?

## 6. Evaluation

Accuracy alone hides *where* a model fails. We look at the per-class report and a confusion matrix to see which specialties get mixed up - and whether those confusions are clinically sensible.

In [ ]:
from sklearn.metrics import classification_report

# Pick the best model by macro-F1
best_name = max(results, key=lambda k: results[k]["macro_f1"])
best = results[best_name]
print(f"Best model: {best_name}\n")

# print a full classification_report
print(classification_report(y_test, best["preds"]))   # TODO

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test, best["preds"],
    xticks_rotation="vertical", cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title(f"Confusion matrix - {best_name}")
plt.tight_layout()
plt.show()

#### Questions for Exploration
1. Find the two specialties that are most often confused with each other. Does that confusion make clinical sense (e.g. overlapping vocabulary)?
2. Which specialty has the lowest recall? Look back at Section 3 - is its training-set size part of the story?
3. `macro-F1` averages F1 across classes equally; `weighted-F1` weights by class size. For a clinical triage tool, which would you report to a stakeholder, and why?

## 7. Extensions

The core pipeline above is deliberately "classical" (TF-IDF + linear model). These optional extensions push toward modern NLP. Some require extra installs and downloads - run only what your environment allows.

### 7a. Handling Imbalance Explicitly

We already used `class_weight="balanced"`. Here you can quantify its effect: train a Logistic Regression **with** and **without** balancing and compare macro-F1. 

In [ ]:
plain    = LogisticRegression(max_iter=1000)                          # no balancing
balanced = LogisticRegression(max_iter=1000, class_weight="balanced")  # with balancing

for label, m in [("plain", plain), ("balanced", balanced)]:
    m.fit(X_train_tfidf, y_train)
    p = m.predict(X_test_tfidf)
    print(f"{label:9s}  acc={accuracy_score(y_test, p):.3f}  macro-F1={f1_score(y_test, p, average='macro'):.3f}")

### 7b. Learned Features: Sentence Embeddings

TF-IDF treats words as independent tokens and ignores meaning ("MI" vs. "heart attack" are unrelated to it). **Embeddings** map text into a dense semantic space. This is the "handcrafted vs. learned features" contrast from the lecture.

> Requires: `pip install sentence-transformers` (downloads a ~90 MB model on first run). If you cannot install it, read the code and skip execution.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

encoder = SentenceTransformer("all-MiniLM-L6-v2")
# Embed the raw (uncleaned) transcriptions - the model handles casing/punctuation itself.
X_embedding_train = encoder.encode(list(X_train), show_progress_bar=True)
X_embedding_test  = encoder.encode(list(X_test),  show_progress_bar=True)

X_embedding_train

In [ ]:
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_embedding_train, y_train)
y_embedding_preds = clf.predict(X_embedding_test)
print("Embedding + LogReg  macro-F1 =", f1_score(y_test, y_embedding_preds, average="macro"))
#

### 8. LLMs for Classification (Gemini API)

A modern alternative is to *prompt* a large language model - no training required. Below we call Google's **Gemini** free tier on a small sample of notes.

> **Privacy:** this sends transcription text to a third-party API. Fine for public `mtsamples` data; never do this with real PHI.

#### Questions for Exploration
1. A zero-shot LLM needs **no labelled training data**. What are two advantages over our TF-IDF model, and two serious risks in a clinical setting (think privacy, hallucination, cost, reproducibility)?
2. How would you *evaluate* the LLM's answers against our test set to compare it fairly with the classifier? (What metric, what ground truth?)
3. When would you still prefer the simple, transparent TF-IDF + Logistic Regression model over an LLM?


In [ ]:
import time
from google import genai
from sklearn.metrics import accuracy_score, f1_score

# Course Gemini free-tier key (shared for this lab)
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"  # free key from https://aistudio.google.com/apikey

client = genai.Client(api_key=GEMINI_API_KEY)
# MODEL = "gemini-flash-latest"  # free-tier friendly alias
MODELS = ["gemini-2.5-flash-lite", "gemma-4-26b", "gemma-4-31b"]

SPECIALTIES = [
    "Cardiovascular / Pulmonary", "Orthopedic", "Radiology",
    "Gastroenterology", "Neurology", "Obstetrics / Gynecology",
    "Urology", "Nephrology", "ENT - Otolaryngology", "Ophthalmology",
]
specialty_list = ", ".join(SPECIALTIES)

def build_prompt(transcription: str) -> str:
    return f"""You are a medical coding assistant. \
        Read the clinical note and respond with exactly \
        one specialty from this list: [{specialty_list}].

Note:
---
{transcription}
---

Specialty:"""

def classify_note(transcription: str) -> str:
    response = client.models.generate_content(
        model=MODEL,
        contents=build_prompt(transcription),
    )
    text = (response.text or "").strip().splitlines()[0].strip()
    # Prefer an exact specialty match if the model adds extra words
    for s in SPECIALTIES:
        if s.lower() in text.lower():
            return s
    return text

# Small sample so we stay within free-tier rate limits
N = 10
sample_X = X_test.head(N)
sample_y = y_test.loc[sample_X.index]

llm_preds = []
for i, note in enumerate(sample_X):
    pred = classify_note(note)
    llm_preds.append(pred)
    print(f"[{i+1}/{N}] true={sample_y.iloc[i]!r:40s}  pred={pred!r}")
    time.sleep(1.0)  # be polite to free-tier RPM limits

acc = accuracy_score(sample_y, llm_preds)
macro_f1 = f1_score(sample_y, llm_preds, average="macro")
print(f"\nGemini zero-shot on n={N}: accuracy={acc:.3f}  macro-F1={macro_f1:.3f}")
print("(This is a tiny sample for demo - not a fair full-test comparison.)")


## Wrap-up

You built an end-to-end clinical-text pipeline: **clean -> TF-IDF -> classify -> evaluate**, dealt with real class imbalance, and looked past classical NLP toward embeddings, NER, and LLMs.

**Key takeaways**
- Unstructured clinical text needs deliberate preprocessing before any model can use it.
- TF-IDF + a linear classifier is a strong, transparent baseline for clinical text.
- On imbalanced clinical data, **macro-F1 and the confusion matrix** tell you far more than accuracy.
- Bigger models (embeddings, LLMs) are not automatically better on small datasets - and they bring privacy, cost, and reproducibility trade-offs, which we revisit in the Bias & Fairness lecture.